# Dynamic Phi Model Training with GRPO

This notebook implements the same training process as the dynamic_phi.py script, allowing for interactive execution and visualization of the training process.

## Setup Logging

First, let's set up logging to track our progress.

In [ ]:
import os

# Set GPU device
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"Using GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


LD_LIBRARY_PATH has been unset
Using GPU: 0

Current environment variables:
PYTHONPATH: /opt/intel/oneapi/advisor/2022.1.0/pythonapi:/usr/local/lib/python2.7/site-packages:/usr/local/lib/python2.7:/usr/local/lib/python2.7/site-packages:/usr/local/lib/python2.7
PATH: /opt/intel/oneapi/intelpython/latest/bin/libfabric:/opt/intel/oneapi/intelpython/latest/bin:/opt/intel/oneapi/intelpython/latest/bin/libfabric:/opt/intel/oneapi/intelpython/latest/bin:/Home/stat/laschos/.vscode-server/cli/servers/Stable-6609ac3d66f4eade5cf376d1cb76f13985724bcb/server/bin/remote-cli:/opt/intel/oneapi/intelpython/latest/bin/libfabric:/Home/stat/laschos/.local/bin:/usr/local/bin:/opt/intel/oneapi/intelpython/latest/bin/libfabric:/Home/stat/laschos/.local/bin:/usr/local/bin:/usr/lib64/mpi/gcc/openmpi2/bin:/opt/intel/oneapi/vtune/2022.3.0/bin64:/opt/intel/oneapi/vpl/2022.1.0/bin:/opt/intel/oneapi/mpi/2021.6.0/libfabric/bin:/opt/intel/oneapi/mpi/2021.6.0/bin:/opt/intel/oneapi/mkl/2022.1.0/bin/intel64:/opt/intel/o

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)
import os
import wandb
import logging
import json
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from datetime import datetime
from unsloth import is_bfloat16_supported
import torch
import sys
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Ensure the project root is in sys.path for imports
import sys
sys.path.append("/Home/stat/laschos/math/AIMO2_initial")
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from grpo.config import RewardConfig
from grpo.dynamic_reward import DynamicReward
from utils.similarity_checker import SolutionSimilarityChecker
from utils.data_preparation import prepare_combined_data
from utils.agents import (
    FULLSOLUTION_SYSTEM_PROMPT, 
    COMPLETION_SYSTEM_PROMPT,
    PROGRAMMER_SYSTEM_PROMPT
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
def setup_logging(model_type: str) -> logging.Logger:
    """Setup logging configuration"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = f"logs/{model_type}"
    os.makedirs(log_dir, exist_ok=True)
    
    logger = logging.getLogger('dynamic_grpo')
    
    # Clear any existing handlers to prevent duplicate logging
    if logger.handlers:
        logger.handlers.clear()
        
    logger.setLevel(logging.INFO)
    
    file_handler = logging.FileHandler(
        f"{log_dir}/training_{timestamp}.log"
    )
    file_handler.setFormatter(logging.Formatter(
        '%(asctime)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
    logger.addHandler(file_handler)
    logger.addHandler(logging.StreamHandler())
    return logger

class LoggingCallback(TrainerCallback):
    """Callback for logging training metrics"""
    def __init__(self, reward_func, logger, save_frequency=100):
        self.reward_func = reward_func
        self.save_frequency = save_frequency
        self.step = 0
        self.logger = logger
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        self.step += 1
        
        if logs and 'rewards/0' in logs and hasattr(self.reward_func, 'stats'):
            # Print detailed stats to console/log file
            self.logger.info("\n" + "="*50)
            self.logger.info(f"Step {self.step} - Reward Stats Summary:")
            
            # Get and log the stats summary
            stats_summary = self.reward_func.stats.get_summary()
            self.logger.info(stats_summary)
            self.logger.info("="*50 + "\n")
            
            # Key performance metrics for wandb
            wandb_stats = {
                'reward': logs['rewards/0'],
                'average_reward': self.reward_func.stats.reward_components.get('average_reward', 0.0),
                'total_batches': self.reward_func.stats.total_batches,
                'total_examples': self.reward_func.stats.total_examples
            }
            
            # Add dynamic reward specific metrics
            if 'solution_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['solution_reward_uses'] = self.reward_func.stats.reward_components['solution_reward_uses']
            if 'completion_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['completion_reward_uses'] = self.reward_func.stats.reward_components['completion_reward_uses']
                
            # Track example types in the batch
            if hasattr(state, 'train_dataloader') and state.train_dataloader is not None:
                try:
                    # Get current batch
                    batch_idx = (state.global_step - 1) % len(state.train_dataloader)
                    current_batch = list(state.train_dataloader)[batch_idx]
                    
                    # Count example types if available
                    if 'example_type' in current_batch:
                        example_types = current_batch['example_type']
                        solution_count = sum(1 for t in example_types if t == 'solution')
                        completion_count = sum(1 for t in example_types if t == 'completion')
                        wait_count = sum(1 for t in example_types if t == 'wait')
                        
                        wandb_stats['solution_examples'] = solution_count
                        wandb_stats['completion_examples'] = completion_count
                        wandb_stats['wait_examples'] = wait_count
                except Exception as e:
                    self.logger.warning(f"Could not track example types: {str(e)}")
            
            # Add all stats from reward_components to wandb
            for key, value in self.reward_func.stats.reward_components.items():
                wandb_stats[f'reward_components/{key}'] = value
                
            # Add group stats
            for key, value in self.reward_func.stats.group_stats.items():
                wandb_stats[f'group_stats/{key}'] = value
                
            # Add step stats
            for key, value in self.reward_func.stats.step_stats.items():
                wandb_stats[f'step_stats/{key}'] = value
                
            # Add similarity stats
            for key, value in self.reward_func.stats.similarity_stats.items():
                wandb_stats[f'similarity_stats/{key}'] = value
                
            # Add programming stats
            for key, value in self.reward_func.stats.programming_stats.items():
                wandb_stats[f'programming_stats/{key}'] = value
                
            # Add reward distribution
            if hasattr(self.reward_func.stats, 'reward_distribution') and self.reward_func.stats.reward_distribution:
                # Only log the top 10 most common rewards to avoid cluttering wandb
                sorted_rewards = sorted(
                    self.reward_func.stats.reward_distribution.items(), 
                    key=lambda x: self.reward_func.stats.reward_distribution[x[0]], 
                    reverse=True
                )[:10]
                
                for reward, count in sorted_rewards:
                    wandb_stats[f'reward_distribution/{reward}'] = count
            
            # Update logs with our metrics
            logs.update(wandb_stats)

Notebook started


## Main Training Setup

Now let's set up the main training configuration and components.

In [ ]:
# Configuration
# Configuration
model_type = "dynamic_0"
model_name = "/Home/stat/laschos/math/AIMO2_initial/models/wait_2/20250304_194943"
dataset_name = "Metaskepsis/completion"

# Setup logging first
logger = setup_logging(model_type)

# Initialize config
reward_config = RewardConfig(model_type=model_type)
reward_config.group_diversity_bonus = 2  # Increased from 1.0

# Setup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"train_results/{reward_config.model_type}/{timestamp}"
wandbname = f"{model_type}, DB={reward_config.group_diversity_bonus}, {model_name}, {dataset_name}, {timestamp}"

# Initialize wandb
wandb.init(
    project="grpo",
    name=wandbname,
    config={
        "model_type": reward_config.model_type,
        "dataset": dataset_name,
        "base_reward": 3.0,
        "diversity_bonus": 0.3,
        "step_continuity_reward": 0.5
    }
)

# Initialize similarity checker first
similarity_checker = SolutionSimilarityChecker(reward_config)

# Initialize dynamic reward function
reward_func = DynamicReward(reward_config, similarity_checker)
logger.info("\nInitialized DynamicReward:")
logger.info(f"Has stats object: {hasattr(reward_func, 'stats')}")

# Print initial stats configuration
if hasattr(reward_func, 'stats'):
    logger.info("Initial stats configuration:")
    for category in ['reward_components', 'group_stats', 'step_stats', 'similarity_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            logger.info(f"{category}: {stats_dict}")
else:
    logger.warning("No stats object found in reward_func!")


Initialized DynamicReward:
Has stats object: True
Initial stats configuration:
reward_components: {'base_rewards': 0, 'step_continuity_rewards': 0, 'diversity_bonuses': 0, 'similarity_penalties': 0, 'validation_rewards': 0, 'total_length_penalty': 0.0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_rewards': 0.0, 'average_reward': 0.0, 'solution_reward_uses': 0, 'completion_reward_uses': 0, 'programming_reward_uses': 0, 'wait_examples_processed': 0, 'wait_examples_rewarded': 0, 'structure_rewards': 0, 'syntax_rewards': 0, 'execution_rewards': 0, 'correctness_rewards': 0, 'syntax_valid_solutions': 0, 'execution_valid_solutions': 0}
group_stats: {'unique_solutions': 0, 'similar_solutions': 0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_similarity': 0.0, 'diversity_bonuses': 0, 'similarity_penalties': 0}
step_stats: {'correct_step_numbering': 0, 'incorrect_step_numbering': 0, 'total_steps_completed': 0}
similarity_stats: {'unique_completions': 0, 'similar_completions': 0, 

In [ ]:
# Load model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=4596,
        fast_inference=True,
        load_in_4bit=False,
        use_gradient_checkpointing="unsloth",
        gpu_memory_utilization=0.6,
        max_lora_rank=64)
        
    # Function to count tokens in a string
    def count_tokens(text):
        return len(tokenizer.encode(text))
        
    # Calculate token counts for system prompts
    solver_prompt_tokens = count_tokens(FULLSOLUTION_SYSTEM_PROMPT)
    completion_prompt_tokens = count_tokens(COMPLETION_SYSTEM_PROMPT)
    logger.info(f"Solver system prompt: {solver_prompt_tokens} tokens")
    logger.info(f"Completion system prompt: {completion_prompt_tokens} tokens")
    
    # Configure LoRA
    model = FastLanguageModel.get_peft_model(
        model,
        r=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
        lora_alpha=64,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None
    )
        
    def get_questions(split="train") -> Dataset:
        """Load and format dataset with full solution, completion, programming, and wait examples
        with the following distribution:
        - 35% solution examples
        - 35% programming examples
        - 15% completion examples
        - 15% wait examples
        """
        
        
        # Load the base dataset
        data = load_dataset(dataset_name, split=split)
        
        # Define the distribution
        distribution = {
            'solution': 0.35,
            'programming': 0.35,
            'completion': 0.15,
            'wait': 0.15
        }
        
        # Use the prepare_combined_data function with programming system prompt
        return prepare_combined_data(
            data, 
            FULLSOLUTION_SYSTEM_PROMPT, 
            COMPLETION_SYSTEM_PROMPT, 
            PROGRAMMER_SYSTEM_PROMPT,
            tokenizer, 
            distribution)

    # Get the formatted dataset with all types of examples
    formatted_dataset = get_questions()
    # Shuffle the combined dataset
    formatted_dataset = formatted_dataset.shuffle(seed=20)
    # Use a reasonable number of examples
    formatted_dataset = formatted_dataset.select(range(2000))
   
    # Verify first few entries
    solution_count = 0
    completion_count = 0
    wait_count = 0
    programming_count = 0
    
    for i in range(min(12, len(formatted_dataset))):
        entry = formatted_dataset[i]
        example_type = entry.get('example_type', 'unknown')
        
        if example_type == 'solution':
            solution_count += 1
        elif example_type == 'completion':
            completion_count += 1
        elif example_type == 'wait':
            wait_count += 1
        elif example_type == 'programming':
            programming_count += 1
            
        print(f"\nEntry {i} verification:")
        print(f"Type: {example_type}")
        print(f"Answer: {entry.get('answer')}")
        
        # Get token count for the prompt
        prompt = entry.get('prompt', '')
        prompt_tokens = count_tokens(prompt)
        print(f"Prompt tokens: {prompt_tokens}")
        
        if example_type == 'completion' and entry.get('partial_solution'):
            partial = entry.get('partial_solution')
            # Count steps in partial solution
            step_count = len(re.findall(r'<step>', partial))
            print(f"Steps in partial solution: {step_count}")
            
        elif example_type == 'wait':
            # Extract thinking section to verify wait modification
            thinking_pattern = re.compile(r'<thinking>(.*?)</thinking>', re.DOTALL)
            thinking_match = thinking_pattern.search(prompt)
        
        # Check for prompt indicators
        has_continue = 'continue' in prompt.lower()
        has_next_step = 'next step' in prompt.lower()
        has_wait = 'wait a second' in prompt.lower()
        print(f"Prompt indicators: continue={has_continue}, next_step={has_next_step}, wait={has_wait}")
    
    print(f"\nSample ratio: {solution_count} solution examples, {completion_count} completion examples, {wait_count} wait examples, {programming_count} programming examples")
    
    # GRPO specific training arguments
    training_args = GRPOConfig(
        torch_empty_cache_steps=1,
        learning_rate=6e-6,
        adam_beta1=0.9,
        adam_beta2=0.99,
        weight_decay=0.1,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        logging_steps=1,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        num_generations=8,
        max_prompt_length=2048,
        max_completion_length=2548,
        num_train_epochs=1,
        save_steps=50,
        max_grad_norm=0.1,
        report_to="wandb",
        output_dir=output_dir,
    )
    
    # Log the dataset structure before training
    logger.info("Dataset structure before training:")
    sample_example = formatted_dataset[0]
    for key, value in sample_example.items():
        logger.info(f"  {key}: {type(value)} - {value}")
    
    # Initialize trainer with reward function
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[reward_func],
        args=training_args,
        train_dataset=formatted_dataset,
        callbacks=[LoggingCallback(reward_func=reward_func, logger=logger, save_frequency=10)]
    )
    
    # Log dataset information before training
    logger.info("Dataset information before training:")
    logger.info(f"Total examples: {len(formatted_dataset)}")
    
    # Count example types in the dataset
    example_types = {}
    for example in formatted_dataset:
        et = example.get('example_type', 'unknown')
        example_types[et] = example_types.get(et, 0) + 1
    
    logger.info(f"Example types in dataset: {example_types}")
    
    # Log a sample batch structure
    sample_batch = {
        'prompt': [formatted_dataset[i]['prompt'] for i in range(min(3, len(formatted_dataset)))],
        'answer': [formatted_dataset[i]['answer'] for i in range(min(3, len(formatted_dataset)))],
        'example_type': [formatted_dataset[i]['example_type'] for i in range(min(3, len(formatted_dataset)))]
    }
    
    logger.info("Sample batch structure:")
    for key, value in sample_batch.items():
        if key != 'prompt':  # Skip logging the full prompts
            logger.info(f"  {key}: {value}")
    
    # The example_type is already in the dataset, no need to add it again
    # Just verify that it's present in all examples
    example_type_missing = sum(1 for example in formatted_dataset if "example_type" not in example)
    if example_type_missing > 0:
        logger.warning(f"Found {example_type_missing} examples without example_type field")
    else:
        logger.info("All examples have example_type field correctly set")
    
    # Print a few examples to verify example_type is set correctly
    for i in range(min(5, len(formatted_dataset))):
        logger.info(f"Example {i} type: {formatted_dataset[i]['example_type']}")

PyTorch version: 2.5.1+cu124
CUDA available: True
CUDA version: 12.4
Current device: 0
Device name: NVIDIA A100-SXM4-40GB


True

## Initialize Trainer

Now let's initialize the GRPO trainer with our model, dataset, and reward function.

## Start Training

Now let's start the training process.

In [ ]:
 # Train
    try:
        trainer.train()
        logger.info("Training completed successfully")
    except Exception as e:
        logger.error(f"Training failed: {str(e)}")
        wandb.finish()
        raise

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 161,480,704/7,777,097,216 (2.08% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: artnoage (metaskepsis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Available kwargs: ['prompts', 'id', 'data_type', 'problem', 'correct_solution', 'correct_answer', 'model_solution', 'model_answer', 'is_correct', 'attempt_number', 'total_attempts', 'answer', 'partial_solution', 'example_type']
example_type found: ['programming', 'programming', 'programming', 'programming', 'programming', 'programming', 'programming', 'programming'] (type: <class 'list'>)
example_type list length: 8
First element: programming (type: <class 'str'>)
Extracted example types: {'programming': 8}
Type counts in batch: completion=0, solution=0, wait=0, programming=8
Selected programming reward (majority type)
Using programming reward for entire batch of 8 examples
Error during batch processing: This event loop is already running
/Home/stat/laschos/math/AIMO2_initial/grpo/dynamic_reward.py:330: RuntimeWarning: coroutine 'DynamicReward.__call__.<locals>.process_batch' was never awaited
  rewards = [0.0] * len(completions)
Rewards before: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

Unsloth: Will smartly offload gradients to save VRAM!


Available kwargs: ['prompts', 'id', 'data_type', 'problem', 'correct_solution', 'correct_answer', 'model_solution', 'model_answer', 'is_correct', 'attempt_number', 'total_attempts', 'answer', 'partial_solution', 'example_type']
example_type found: ['completion', 'completion', 'completion', 'completion', 'completion', 'completion', 'completion', 'completion'] (type: <class 'list'>)
example_type list length: 8
First element: completion (type: <class 'str'>)
Extracted example types: {'completion': 8}
Type counts in batch: completion=8, solution=0, wait=0, programming=0
Selected completion reward (majority type)
Using completion reward for entire batch of 8 examples
Error during batch processing: This event loop is already running
Rewards before: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

Reward Statistics Summary:
Training time: 0:07:51.113090
Processed 6 batches (24 examples)
Average reward: 0.000000
Reward range: [0.0000, 0.0000]

Reward Distribution:
  -0.10:    0 |
  -0.06:    0 |
  -0

Step,Training Loss,reward,reward_std,completion_length,kl,rewards / dynamic_reward
1,0.000000,0.000000,0.000000,736.125000,0.000000,0.000000
2,0.000000,0.000000,0.000000,791.218750,0.000000,0.000000
3,0.000000,0.000000,0.000000,543.062500,0.000000,0.000000
4,0.000000,0.000000,0.000000,948.031250,0.000000,0.000000
5,0.000000,0.000000,0.000000,536.312500,0.000000,0.000000
6,0.000000,0.000000,0.000000,524.937500,0.000000,0.000000
7,0.000000,0.000000,0.000000,500.906250,0.000000,0.000000
8,0.000000,0.000000,0.000000,498.562500,0.000000,0.000000
9,0.000000,0.000000,0.000000,659.468750,0.000000,0.000000
10,0.000000,0.000000,0.000000,680.312500,0.000000,0.000000


Available kwargs: ['prompts', 'id', 'data_type', 'problem', 'correct_solution', 'correct_answer', 'model_solution', 'model_answer', 'is_correct', 'attempt_number', 'total_attempts', 'answer', 'partial_solution', 'example_type']
example_type found: ['completion', 'completion', 'completion', 'completion', 'completion', 'completion', 'completion', 'completion'] (type: <class 'list'>)
example_type list length: 8
First element: completion (type: <class 'str'>)
Extracted example types: {'completion': 8}
Type counts in batch: completion=8, solution=0, wait=0, programming=0
Selected completion reward (majority type)
Using completion reward for entire batch of 8 examples
Error during batch processing: This event loop is already running
Rewards before: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

Reward Statistics Summary:
Training time: 0:09:13.293392
Processed 10 batches (40 examples)
Average reward: 0.000000
Reward range: [0.0000, 0.0000]

Reward Distribution:
  -0.10:    0 |
  -0.06:    0 |
  -

## Visualize Training Results

Let's visualize the training metrics.

In [ ]:
# Plot the metrics
notebook_callback.plot_metrics()

# Print final reward statistics
if hasattr(reward_func, 'stats'):
    print("\nFinal Reward Statistics Summary:")
    print(reward_func.stats.get_summary())
    
    print("\nReward Components:")
    for key, value in reward_func.stats.reward_components.items():
        print(f"  {key}: {value}")
    
    # Check for other stat categories
    for category in ['group_stats', 'step_stats', 'similarity_stats', 'programming_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            if stats_dict:
                print(f"\n{category.replace('_', ' ').title()}:")
                for key, value in stats_dict.items():
                    print(f"  {key}: {value}")
else:
    print("No statistics available.")

## Save Model

Finally, let's save the trained model.

In [ ]:
# Save model
try:
    models_dir = "models"
    os.makedirs(os.path.join(models_dir, reward_config.model_type), exist_ok=True)
    model_output_dir = os.path.join(models_dir, reward_config.model_type, timestamp)
    model.save_pretrained_merged(model_output_dir, tokenizer, save_method="merged_16bit")
    logger.info(f"Merged model saved to {model_output_dir}")
    print(f"Model saved to {model_output_dir}")
except Exception as e:
    logger.error(f"Failed to save model: {str(e)}")
    print(f"Error saving model: {str(e)}")
finally:
    if use_wandb:
        wandb.finish()
        print("Wandb logging finished")

## Conclusion

This notebook has demonstrated the complete training process for a Qwen model using GRPO with dynamic rewards. We've seen how to:

1. Configure the reward function
2. Prepare a dataset with different example types
3. Initialize and configure the model with LoRA
4. Set up the GRPO training process
5. Train the model and visualize the results
6. Save the trained model

This interactive approach allows for better monitoring and understanding of the training process compared to running the script directly.